# HumanForYou - Quels facteurs expliquent les departs ? Optimisation et recommandations

Le notebook precedent a montre que le Random Forest est le meilleur modele. Mais un modele qui predit sans expliquer ne repond qu'a la moitie de la question posee par HumanForYou.

La direction veut savoir **pourquoi** ses employes partent, pas seulement **lesquels** vont partir. Ce notebook repond a cette question en deux temps :
1. On optimise le RF pour avoir les meilleures predictions possibles
2. On analyse ce que le modele nous dit sur les facteurs de depart, avec des chiffres concrets issus des donnees brutes

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import warnings

from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble         import RandomForestClassifier
from sklearn.metrics          import (classification_report, roc_auc_score,
                                      f1_score, ConfusionMatrixDisplay, roc_curve)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

RAW    = "../data/raw/"
PROC   = "../data/processed/"
MODELS = "../models/" 

## 1. Chargement des donnees pour la modelisation

In [ ]:
df = pd.read_csv(PROC + "dataset_final.csv")

X = df.drop(columns=["Attrition", "EmployeeID"])
y = df["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")
print(f"Partants dans le test : {y_test.sum()} sur {len(y_test)} ({y_test.mean()*100:.1f}%)")

## 2. Point de depart : RF avec les parametres par defaut

On reentrainele RF de base pour avoir une reference claire avant optimisation.

In [ ]:
rf_base = RandomForestClassifier(
    n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1
)
rf_base.fit(X_train, y_train)

y_pred_base  = rf_base.predict(X_test)
y_proba_base = rf_base.predict_proba(X_test)[:, 1]

f1_base  = f1_score(y_test, y_pred_base)
auc_base = roc_auc_score(y_test, y_proba_base)

print(f"RF de base  ->  F1 : {f1_base:.4f} | AUC : {auc_base:.4f}")

## 3. Recherche des meilleurs hyperparametres

Les hyperparametres, c'est ce qu'on regle avant l'entrainement :
- `n_estimators` : combien d'arbres dans la foret
- `max_depth` : jusqu'ou chaque arbre peut pousser (None = pas de limite)

On teste 8 combinaisons et on evalue chacune par validation croisee a 5 folds. C'est plus transparent que de laisser une fonction automatique fouiller dans tous les sens.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = [
    {"n_estimators": 100, "max_depth": None},
    {"n_estimators": 100, "max_depth": 20},
    {"n_estimators": 200, "max_depth": None},
    {"n_estimators": 200, "max_depth": 20},
    {"n_estimators": 300, "max_depth": None},
    {"n_estimators": 300, "max_depth": 20},
    {"n_estimators": 300, "max_depth": 30},
    {"n_estimators": 500, "max_depth": 30},
]

resultats = []
print(f"  {'n_estimators':>15} {'max_depth':>12} {'F1 moyen':>12} {'Std':>8}")
print("  " + "-"*52)

for params in grid:
    rf = RandomForestClassifier(
        class_weight="balanced", random_state=42, n_jobs=-1, **params
    )
    scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring="f1")
    resultats.append({**params, "f1_mean": scores.mean(), "f1_std": scores.std()})
    print(f"  {params['n_estimators']:>15} {str(params['max_depth']):>12} {scores.mean():>12.4f} {scores.std():>8.4f}")

resultats_df = pd.DataFrame(resultats)
best_idx = resultats_df["f1_mean"].idxmax()
best_params = resultats_df.loc[best_idx]
print(f"\nMeilleure config : n_estimators={int(best_params['n_estimators'])}, max_depth={best_params['max_depth']}")
print(f"F1 moyen en CV   : {best_params['f1_mean']:.4f} +/- {best_params['f1_std']:.4f}")

## 4. Modele final optimise

In [ ]:
max_d = None if pd.isna(best_params["max_depth"]) else int(best_params["max_depth"])

rf_opti = RandomForestClassifier(
    n_estimators=int(best_params["n_estimators"]),
    max_depth=max_d,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_opti.fit(X_train, y_train)

y_pred_opti  = rf_opti.predict(X_test)
y_proba_opti = rf_opti.predict_proba(X_test)[:, 1]

f1_opti  = f1_score(y_test, y_pred_opti)
auc_opti = roc_auc_score(y_test, y_proba_opti)

print("=== Comparaison avant / apres optimisation ===")
print(f"{'RF de base':<35} F1 = {f1_base:.4f} | AUC = {auc_base:.4f}")
print(f"{'RF optimise':<35} F1 = {f1_opti:.4f} | AUC = {auc_opti:.4f}")
print()
print("--- Rapport complet (modele optimise) ---")
print(classification_report(y_test, y_pred_opti, target_names=["Reste (0)", "Part (1)"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_opti,
    display_labels=["Reste", "Part"],
    cmap="Blues", ax=axes[0]
)
axes[0].set_title("Matrice de confusion - RF optimise", fontweight="bold")

for label, y_proba, color in [
    ("RF de base",  y_proba_base, "steelblue"),
    ("RF optimise", y_proba_opti, "seagreen")
]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    axes[1].plot(fpr, tpr, label=f"{label} (AUC={auc:.3f})", color=color)

axes[1].plot([0, 1], [0, 1], "k--", label="Aleatoire")
axes[1].set_xlabel("Taux faux positifs")
axes[1].set_ylabel("Rappel")
axes[1].set_title("Courbes ROC - avant / apres optimisation", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("../outputs/rf_optimise_evaluation.png", bbox_inches="tight")
plt.show()

## 5. Quels facteurs expliquent les departs ?

C'est la question centrale du projet. Le Random Forest calcule nativement l'importance de chaque variable via la reduction moyenne de l'impurete de Gini : plus une variable est utilisee pour separer efficacement les partants des restants dans les arbres, plus son score est eleve.

Ce graphique repond directement a la demande de HumanForYou : **sur quoi agir pour reduire le taux de rotation de 15% ?**

In [ ]:
importances = pd.Series(rf_opti.feature_importances_, index=X.columns)
top20 = importances.sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 7))
top20.sort_values().plot(kind="barh", color="steelblue")
plt.xlabel("Importance (reduction Gini moyenne)")
plt.title("Facteurs les plus predictifs du depart - RF optimise", fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/feature_importance.png", bbox_inches="tight")
plt.show()

print("Top 10 des facteurs de depart :")
for rang, (feat, imp) in enumerate(top20.head(10).items(), 1):
    print(f"  {rang:2}. {feat:<35} {imp:.4f}")

## 6. Combien de difference concretement ?

L'importance des variables nous dit *quelles* variables comptent. Maintenant on regarde les valeurs brutes pour comprendre *dans quel sens* et *de combien*. On charge les fichiers source originaux (non normalises) pour avoir des chiffres interpretables : roupies, annees, jours.

In [ ]:
# Chargement des donnees brutes (non normalisees)
general = pd.read_csv(RAW + "general_data.csv")
survey  = pd.read_csv(RAW + "employee_survey_data.csv")
manager = pd.read_csv(RAW + "manager_survey_data.csv")

# Calcul des features de badgeage
in_time  = pd.read_csv(RAW + "in_time.csv",  index_col=0)
out_time = pd.read_csv(RAW + "out_time.csv", index_col=0)
in_dt    = in_time.apply(pd.to_datetime, errors="coerce")
out_dt   = out_time.apply(pd.to_datetime, errors="coerce")
work_hours = (out_dt - in_dt).apply(lambda col: col.dt.total_seconds() / 3600)

badge = pd.DataFrame({
    "EmployeeID"       : general["EmployeeID"].values,
    "avg_hours_per_day": work_hours.mean(axis=1).values,
    "days_absent"      : work_hours.isna().sum(axis=1).values,
    "days_over_8h"     : (work_hours > 8).sum(axis=1).values,
})

# Fusion des 4 sources sur EmployeeID
brut = general.merge(survey,  on="EmployeeID", how="left")
brut = brut.merge(manager, on="EmployeeID", how="left")
brut = brut.merge(badge,   on="EmployeeID", how="left")

# Imputation mediane (coherent avec le preprocessing)
for col in brut.select_dtypes(include="number").columns:
    brut[col] = brut[col].fillna(brut[col].median())

restants = brut[brut["Attrition"] == "No"]
partants = brut[brut["Attrition"] == "Yes"]

print(f"Restants : {len(restants)} | Partants : {len(partants)}")

In [ ]:
vars_comparaison = {
    "MonthlyIncome"          : "Salaire mensuel (roupies)",
    "YearsAtCompany"         : "Anciennete (annees)",
    "YearsSinceLastPromotion": "Annees sans promotion",
    "TotalWorkingYears"      : "Annees experience totale",
    "DistanceFromHome"       : "Distance domicile-travail (km)",
    "TrainingTimesLastYear"  : "Formations en 2015 (jours)",
    "avg_hours_per_day"      : "Heures travaillees / jour",
    "days_absent"            : "Jours d'absence en 2015",
    "days_over_8h"           : "Jours depasses 8h",
    "EnvironmentSatisfaction": "Satisfaction environnement (1-4)",
    "JobSatisfaction"        : "Satisfaction au travail (1-4)",
    "WorkLifeBalance"        : "Equilibre vie pro/perso (1-4)",
    "JobInvolvement"         : "Implication au travail (1-4)",
    "StockOptionLevel"       : "Investissement actions (0-3)",
}

lignes = []
for col, label in vars_comparaison.items():
    if col not in brut.columns:
        continue
    m_r = restants[col].mean()
    m_p = partants[col].mean()
    delta = m_p - m_r
    delta_pct = (delta / m_r * 100) if m_r != 0 else 0
    lignes.append({
        "Variable"     : label,
        "Restants (moy)": round(m_r, 1),
        "Partants (moy)": round(m_p, 1),
        "Difference"   : round(delta, 1),
        "Variation"    : f"{delta_pct:+.1f}%"
    })

compar = pd.DataFrame(lignes).set_index("Variable")
print(compar.to_string())

In [ ]:
# Visualisation des 6 variables les plus discriminantes (valeurs brutes)
vars_plot = [
    ("MonthlyIncome",           "Salaire mensuel (roupies)"),
    ("YearsAtCompany",          "Anciennete (annees)"),
    ("YearsSinceLastPromotion", "Annees sans promotion"),
    ("JobSatisfaction",         "Satisfaction travail (1-4)"),
    ("EnvironmentSatisfaction", "Satisfaction environnement (1-4)"),
    ("days_absent",             "Jours d'absence 2015"),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, (col, label) in enumerate(vars_plot):
    data_r = restants[col]
    data_p = partants[col]
    axes[i].hist(data_r, bins=20, alpha=0.6, label="Reste", color="steelblue", density=True)
    axes[i].hist(data_p, bins=20, alpha=0.6, label="Part",  color="tomato",    density=True)
    axes[i].set_title(label, fontweight="bold")
    axes[i].legend()

plt.suptitle("Distribution des variables cles : qui reste vs qui part (valeurs brutes)", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("../outputs/profil_partants_vs_restants.png", bbox_inches="tight")
plt.show()

## 7. Recommandations pour HumanForYou

Chaque recommandation est directement fondee sur un ecart mesure dans les donnees et confirme par l'importance calculee par le modele.

In [ ]:
# On recupere les chiffres cles pour les afficher dans les recommandations
inc_r   = restants["MonthlyIncome"].mean()
inc_p   = partants["MonthlyIncome"].mean()
promo_r = restants["YearsSinceLastPromotion"].mean()
promo_p = partants["YearsSinceLastPromotion"].mean()
sat_r   = restants["JobSatisfaction"].mean()
sat_p   = partants["JobSatisfaction"].mean()
env_r   = restants["EnvironmentSatisfaction"].mean()
env_p   = partants["EnvironmentSatisfaction"].mean()
abs_r   = restants["days_absent"].mean()
abs_p   = partants["days_absent"].mean()
train_r = restants["TrainingTimesLastYear"].mean()
train_p = partants["TrainingTimesLastYear"].mean()
dist_r  = restants["DistanceFromHome"].mean()
dist_p  = partants["DistanceFromHome"].mean()

recommandations = [
    (
        "Revalorisation salariale ciblee",
        "MonthlyIncome",
        (f"Restants : {inc_r:.0f} roupies/mois | Partants : {inc_p:.0f} roupies/mois ({(inc_p-inc_r)/inc_r*100:+.1f}%)"
         " -> Identifier les employes en dessous de la mediane salariale de leur niveau"
         " et les revaloriser en priorite avant qu'ils ne cherchent ailleurs.")
    ),
    (
        "Gestion des carrieres et promotions",
        "YearsSinceLastPromotion",
        (f"Restants : {promo_r:.1f} ans sans promotion | Partants : {promo_p:.1f} ans ({(promo_p-promo_r)/promo_r*100:+.1f}%)"
         " -> Mettre en place une revue annuelle : tout employe sans evolution sur 3 ans"
         " doit avoir un plan de developpement concret.")
    ),
    (
        "Ameliorer la satisfaction au travail et l'environnement",
        "JobSatisfaction + EnvironmentSatisfaction",
        (f"Satisfaction travail - Restants : {sat_r:.2f}/4 | Partants : {sat_p:.2f}/4"
         f" | Environnement - Restants : {env_r:.2f}/4 | Partants : {env_p:.2f}/4"
         " -> Rendre l'enquete de satisfaction trimestrielle et declencher des actions"
         " correctrices des qu'un service passe sous 2.5/4.")
    ),
    (
        "Suivi de l'absenteisme comme signal d'alerte precoce",
        "days_absent",
        (f"Restants : {abs_r:.1f} jours | Partants : {abs_p:.1f} jours ({(abs_p-abs_r)/abs_r*100:+.1f}%)"
         " -> Declencher un entretien RH des qu'un employe depasse 30 jours d'absence sur l'annee.")
    ),
    (
        "Investissement en formation",
        "TrainingTimesLastYear",
        (f"Restants : {train_r:.2f} formations/an | Partants : {train_p:.2f} formations/an"
         " -> S'assurer qu'aucun employe ne passe une annee sans formation.")
    ),
    (
        "Flexibilite pour les employes habitant loin",
        "DistanceFromHome",
        (f"Restants : {dist_r:.1f} km | Partants : {dist_p:.1f} km"
         " -> Proposer du teletravail partiel ou des aides a la mobilite pour les employes"
         " dont le trajet depasse 20 km.")
    ),
    (
        "Systeme d'alerte precoce base sur le modele RF",
        "Probabilite de depart (modele)",
        (f"Le modele predit les departs avec AUC = {auc_opti:.3f} sur le jeu de test."
         " -> Utiliser le score de probabilite mensuel pour cibler les entretiens RH"
         " sur les employes a risque (probabilite > 60%) avant que la decision soit prise.")
    ),
]

for i, (titre, source, desc) in enumerate(recommandations, 1):
    print("
" + "="*65)
    print(f"Recommandation {i} : {titre}")
    print(f"Variable(s) : {source}")
    print("="*65)
    print(desc)

## 8. Sauvegarde du modele final

In [ ]:
os.makedirs(MODELS, exist_ok=True)

joblib.dump(rf_opti, MODELS + "random_forest_final.pkl")
joblib.dump(list(X.columns), MODELS + "feature_columns.pkl")

print(f"Modele sauvegarde     -> {MODELS}random_forest_final.pkl")
print(f"Features sauvegardees -> {MODELS}feature_columns.pkl")
print(f"Nombre de features : {len(X.columns)}")
print(f"Parametres : n_estimators={rf_opti.n_estimators}, max_depth={rf_opti.max_depth}")
print(f"\nF1 final sur test  : {f1_opti:.4f}")
print(f"AUC final sur test : {auc_opti:.4f}")